Question 2A.2

In [2]:
from neo4j import GraphDatabase
uri = "neo4j+s://a21d3c40.databases.neo4j.io"
user = "neo4j"
password = "rh0lkCJI9pOZIwIFcW0Qa7Yc9JJcG5vkue0pdsWIzSo"

driver = GraphDatabase.driver(uri, auth=(user, password))

def run_query(query, params=None):
    with driver.session() as session:
        return list(session.run(query, params or {}))

if __name__ == "__main__":
    try:
        result = run_query("RETURN 1 AS ok")
        print("Connection OK:", result)
    except Exception as e:
        print("ERROR:", e)

Connection OK: [<Record ok=1>]


In [5]:
has_genre = """
MATCH (m:Movie)
UNWIND [
['unknown', m.unknown],
['Action', m.Action],
['Adventure', m.Adventure],
['Animation', m.Animation],
["Children's",m.Children],
['Comedy', m.Comedy],
['Crime', m.Crime],
['Documentary',m.Documentary],
['Drama', m.Drama],
['Fantasy', m.Fantasy],
['Film-Noir', m.FilmNoir],
['Horror', m.Horror],
['Musical', m.Musical],
['Mystery', m.Mystery],
['Romance', m.Romance],
['Sci-Fi', m.SciFi],
['Thriller', m.Thriller],
['War', m.War],
['Western', m.Western]
] AS pair
WITH m, pair[0] AS genreName, pair[1] AS flag
WHERE flag IN [1, '1', true]
MATCH (g:Genre {genreName: genreName})
MERGE (m)-[:HAS_GENRE]->(g);
"""
with driver.session() as session:
    session.run(has_genre)


Question 2B

In [6]:
recommendation = """
// Recommend 10 highly rated movies for user 4 that they have not watched
MATCH (u:User {userId: 4})-[:RATED]->(m:Movie)
MATCH (u)-[:RATED]->(m)<-[:RATED]-(other:User)
WHERE other <> u

MATCH (other)-[r:RATED]->(rec:Movie)
WHERE r.rating >= 4
// Excludes movies the user has already rated
AND NOT (u)-[:RATED]->(rec)

// Average rating and how many similar users liked it
RETURN rec.title  AS recommendedMovie,
avg(r.rating)  AS avgRating,
count(DISTINCT other)  AS numSimilarUsers
ORDER BY avgRating DESC, numSimilarUsers DESC
LIMIT 10;
"""

with driver.session() as session:
    session.run(recommendation)

2C

In [20]:
triangle = """
// Movie triangles with shared genre(s) and high ratings
// Restrict to 30 reasonably popular movies, so that code can run without issues with memory
MATCH (m:Movie)<-[r:RATED]-()
WITH m, count(r) AS ratingCount
WHERE ratingCount >= 40
WITH collect(m)[0..30] AS movies

// Generates triples (triangles) of distinct movies
UNWIND movies AS movie1
UNWIND movies AS movie2
UNWIND movies AS movie3
WITH movie1, movie2, movie3
WHERE movie1.movieId < movie2.movieId
  AND movie2.movieId < movie3.movieId

// Making sure all movies in triangle share at least one genre
MATCH (movie1)-[:HAS_GENRE]->(g:Genre)<-[:HAS_GENRE]-(movie2)
MATCH (movie1)-[:HAS_GENRE]->(g:)<-[:HAS_GENRE]-(movie3)

// Find users who rated all three highly (rating >= 4)
MATCH (u:User)-[r1:RATED]->(movie1),
      (u:)-[r2:RATED]->(movie2),
      (u:)-[r3:RATED]->(movie3)
WHERE r1.rating >= 4 AND r2.rating >= 4 AND r3.rating >= 4

RETURN g.genreName,
    movie1.title,
    movie2.title,
    movie3.title,
    count(DISTINCT u) AS numHighRaters
ORDER BY numHighRaters DESC
LIMIT 30;
"""

with driver.session() as session:
    session.run(triangle)

'\n// Movie triangles with shared genre(s) and high ratings\n// Restrict to 30 reasonably popular movies, so that code can run without issues with memory\nMATCH (m:Movie)<-[r:RATED]-()\nWITH m, count(r) AS ratingCount\nWHERE ratingCount >= 40\nWITH collect(m)[0..30] AS movies\n\n// Generates triples (triangles) of distinct movies\nUNWIND movies AS movie1\nUNWIND movies AS movie2\nUNWIND movies AS movie3\nWITH movie1, movie2, movie3\nWHERE movie1.movieId < movie2.movieId\n  AND movie2.movieId < movie3.movieId\n\n// Making sure all movies in triangle share at least one genre\nMATCH (movie1)-[:HAS_GENRE]->(g:Genre)<-[:HAS_GENRE]-(movie2)\nMATCH (movie1)-[:HAS_GENRE]->(g:)<-[:HAS_GENRE]-(movie3)\n\n// Find users who rated all three highly (rating >= 4)\nMATCH (u:User)-[r1:RATED]->(movie1),\n      (u:)-[r2:RATED]->(movie2),\n      (u:)-[r3:RATED]->(movie3)\nWHERE r1.rating >= 4 AND r2.rating >= 4 AND r3.rating >= 4\n\nRETURN g.genreName,\n    movie1.title,\n    movie2.title,\n    movie3.tit

Question 2D

In [7]:
similar = """
// Picks a manageable set of popular movies
MATCH (m:Movie)<-[r:RATED]-()
WITH m, count(r) AS ratingCount
WHERE ratingCount >= 50
ORDER BY ratingCount DESC
WITH collect(m)[0..30] AS movies

// Generates distinct movie pairs
UNWIND movies AS movie1
UNWIND movies AS movie2
WITH movie1, movie2
WHERE movie1.movieId < movie2.movieId  // avoid duplicates and self-pairs

// Compute genreScore, which is the number of shared genres
MATCH (movie1)-[:HAS_GENRE]->(g:Genre)<-[:HAS_GENRE]-(movie2)
WITH movie1, movie2, count(DISTINCT g) AS genreScore

// Keeps only the strong similarities
WHERE genreScore >= 3  // threshold recommended in the correction

// Creates the SIMILAR_TO edge
MERGE (movie1)-[rel:SIMILAR_TO]->(movie2)
SET rel.genreScore = genreScore

RETURN movie1.title,
	movie2.title,
    genreScore
ORDER BY genreScore DESC
LIMIT 20;
"""

with driver.session() as session:
    session.run(similar)
